# ecoli4: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('04_ecoli4_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,89.282,88.571,100.000,80.0,100.000,90.000,96.222,99.524,88.927
smote,88.686,80.286,83.333,80.0,98.730,89.365,95.556,99.444,84.486
bsmote,88.686,80.286,83.333,80.0,98.730,89.365,94.579,99.325,84.486
ros,88.476,78.240,80.119,80.0,98.254,89.127,94.706,99.325,83.358
adasyn,90.945,79.052,76.905,85.0,97.778,91.389,94.968,99.325,84.998
casb,88.686,80.286,83.333,80.0,98.730,89.365,95.079,99.365,84.486
casb_relaxed,88.686,80.286,83.333,80.0,98.730,89.365,95.556,99.444,84.486
casb_edited,90.026,81.714,83.333,82.5,98.730,90.615,95.317,99.405,85.870
casb_ros,88.686,80.286,83.333,80.0,98.730,89.365,95.079,99.365,84.486


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,91.824,89.286,95.000,85.0,99.683,92.341,90.597,94.722,90.555
smote,91.086,80.143,78.333,85.0,98.095,91.548,86.097,94.505,85.614
bsmote,90.879,77.455,72.619,85.0,97.624,91.312,82.264,94.209,84.167
ros,90.809,76.909,71.905,85.0,97.465,91.233,81.847,94.366,83.859
adasyn,93.388,78.798,71.571,90.0,97.309,93.655,82.597,94.248,86.093
casb,90.811,76.621,71.119,85.0,97.468,91.234,84.930,94.387,83.716
casb_relaxed,91.156,80.810,79.333,85.0,98.254,91.627,90.097,94.643,85.983
casb_edited,93.836,83.667,79.333,90.0,98.254,94.127,86.847,94.563,88.751
casb_ros,90.809,76.909,71.905,85.0,97.465,91.233,84.930,94.407,83.859


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,69.641,66.286,90.000,57.5,100.000,78.750,91.944,98.733,67.963
smote,87.237,80.524,88.333,77.5,99.053,88.276,92.756,99.010,83.880
bsmote,85.535,76.714,81.667,75.0,98.735,86.868,85.326,98.655,81.124
ros,78.433,70.603,88.000,65.0,99.211,82.106,89.117,98.813,74.518
adasyn,88.688,80.286,83.333,80.0,98.735,89.368,86.167,98.614,84.487
casb,80.520,73.556,86.333,67.5,99.211,83.356,88.054,98.377,77.038
casb_relaxed,88.826,82.429,88.333,80.0,99.053,89.526,91.694,99.131,85.627
casb_edited,88.826,82.429,88.333,80.0,99.053,89.526,89.708,99.051,85.627
casb_ros,82.161,75.944,90.167,70.0,99.370,84.685,90.269,98.734,79.053


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,91.962,91.429,100.000,85.0,100.000,92.500,97.333,99.762,91.695
smote,91.185,81.609,82.429,85.0,98.418,91.709,96.190,99.603,86.397
bsmote,90.911,77.476,73.667,85.0,97.788,91.394,95.556,99.444,84.194
ros,90.887,78.165,76.762,85.0,97.788,91.394,95.556,99.444,84.526
adasyn,90.547,74.486,69.667,85.0,96.992,90.996,94.500,99.405,82.517
casb,90.819,76.855,73.262,85.0,97.629,91.314,95.456,99.444,83.837
casb_relaxed,90.968,78.577,76.095,85.0,97.944,91.472,96.429,99.643,84.772
casb_edited,90.670,75.294,70.810,85.0,97.314,91.157,95.833,99.524,82.982
casb_ros,90.749,76.022,71.762,85.0,97.470,91.235,94.389,99.247,83.386


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,85.677,83.893,96.250,76.875,99.921,88.398,94.024,98.185,84.785
smote,89.548,80.640,83.107,81.875,98.574,90.224,92.650,98.141,85.094
bsmote,89.003,77.983,77.821,81.250,98.219,89.735,89.431,97.908,83.493
ros,87.151,75.979,79.196,78.750,98.180,88.465,90.307,97.987,81.565
adasyn,90.892,78.155,75.369,85.000,97.703,91.352,89.558,97.898,84.524
casb,87.709,76.829,78.512,79.375,98.260,88.817,90.880,97.894,82.269
casb_relaxed,89.909,80.525,81.774,82.500,98.495,90.498,93.444,98.215,85.217
casb_edited,90.839,80.776,80.452,84.375,98.338,91.356,91.927,98.136,85.808
casb_ros,88.101,77.290,79.292,80.000,98.259,89.129,91.167,97.938,82.696


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,90.026,adasyn,90.945,-0.919,LOSS
1,f1,casb_edited,81.714,smote,80.286,1.429,WIN
2,precision,casb,83.333,smote,83.333,0.000,TIE
3,recall,casb_edited,82.500,adasyn,85.000,-2.500,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,93.836,adasyn,93.388,0.448,WIN
1,f1,casb_edited,83.667,smote,80.143,3.524,WIN
2,precision,casb_relaxed,79.333,smote,78.333,1.000,WIN
3,recall,casb_edited,90.000,adasyn,90.000,0.000,TIE


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,88.826,adasyn,88.688,0.138,WIN
1,f1,casb_relaxed,82.429,smote,80.524,1.905,WIN
2,precision,casb_ros,90.167,smote,88.333,1.833,WIN
3,recall,casb_relaxed,80.000,adasyn,80.000,0.000,TIE


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,90.839,adasyn,90.892,-0.053,LOSS
1,f1,casb_edited,80.776,smote,80.640,0.136,WIN
2,precision,casb_relaxed,81.774,smote,83.107,-1.333,LOSS
3,recall,casb_edited,84.375,adasyn,85.000,-0.625,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.866025,0.857143,1.000000,0.75,1.000000,0.875000,1.000000,1.000000,0.861584,None
1,42,1,svm,smote,0.50,0.866025,0.857143,1.000000,0.75,1.000000,0.875000,1.000000,1.000000,0.861584,None
2,42,1,svm,bsmote,0.50,0.866025,0.857143,1.000000,0.75,1.000000,0.875000,1.000000,1.000000,0.861584,None
3,42,1,svm,ros,0.50,0.866025,0.857143,1.000000,0.75,1.000000,0.875000,1.000000,1.000000,0.861584,None
4,42,1,svm,adasyn,0.50,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.000000,None
5,42,1,svm,casb,0.50,0.866025,0.857143,1.000000,0.75,1.000000,0.875000,1.000000,1.000000,0.861584,None
6,42,1,svm,casb_relaxed,0.50,0.866025,0.857143,1.000000,0.75,1.000000,0.875000,1.000000,1.000000,0.861584,None
7,42,1,svm,casb_edited,0.50,0.866025,0.857143,1.000000,0.75,1.000000,0.875000,1.000000,1.000000,0.861584,None
8,42,1,svm,casb_ros,0.50,0.866025,0.857143,1.000000,0.75,1.000000,0.875000,1.000000,1.000000,0.861584,None
9,42,1,knn,original,0.50,0.866025,0.857143,1.000000,0.75,1.000000,0.875000,1.000000,1.000000,0.861584,None
